In [4]:
import pandas as pd
from pathlib import Path

data_path = Path("../data/processed")

files = {
    "customers": "olist_customers_clean.csv",
    "orders": "olist_orders_clean.csv",
    "order_items": "olist_order_items_clean.csv",
    "order_payments": "olist_order_payments_clean.csv",
    "order_reviews": "olist_order_reviews_clean.csv",
    "products": "olist_products_clean.csv",
    "sellers": "olist_sellers_clean.csv",
}

for table, file in files.items():
    df = pd.read_csv(data_path / file)

    print(f"\n{'='*60}")
    print(f"{table.upper()}")
    print(f"Rows: {len(df):,}")
    print("Columns:")
    print(df.columns.tolist())


CUSTOMERS
Rows: 99,441
Columns:
['customer_id', 'customer_unique_id', 'customer_zip_code_prefix', 'customer_city', 'customer_state']

ORDERS
Rows: 99,441
Columns:
['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date', 'invalid_date_sequence', 'delivery_time_days']

ORDER_ITEMS
Rows: 112,650
Columns:
['order_id', 'order_item_id', 'product_id', 'seller_id', 'shipping_limit_date', 'price', 'freight_value']

ORDER_PAYMENTS
Rows: 103,886
Columns:
['order_id', 'payment_sequential', 'payment_type', 'payment_installments', 'payment_value', 'invalid_installments', 'undefined_zero_payment']

ORDER_REVIEWS
Rows: 99,224
Columns:
['review_id', 'order_id', 'review_score', 'review_comment_title', 'review_comment_message', 'review_creation_date', 'review_answer_timestamp']

PRODUCTS
Rows: 32,951
Columns:
['product_id', 'product_category_name', 'product_name_lenght', 

## 5. Inspect Cleaned Dataset Structure

Before creating PostgreSQL tables, we first inspect the cleaned CSV files.

The objective is to understand:

- Number of records in each dataset
- Column names
- Table-level grain
- Potential primary keys
- Foreign key relationships
- Additional columns created during data cleaning

This step ensures that the PostgreSQL schema is designed according to the actual cleaned data rather than making assumptions about the dataset structure.

In [5]:
for table, file in files.items():
    df = pd.read_csv(data_path / file)

    print(f"\n{'='*70}")
    print(f"{table.upper()}")
    print(f"{'='*70}")

    print(f"Rows: {len(df):,}")
    print(f"Columns ({len(df.columns)}):")

    for i, col in enumerate(df.columns, start=1):
        print(f"{i}. {col}")


CUSTOMERS
Rows: 99,441
Columns (5):
1. customer_id
2. customer_unique_id
3. customer_zip_code_prefix
4. customer_city
5. customer_state

ORDERS
Rows: 99,441
Columns (10):
1. order_id
2. customer_id
3. order_status
4. order_purchase_timestamp
5. order_approved_at
6. order_delivered_carrier_date
7. order_delivered_customer_date
8. order_estimated_delivery_date
9. invalid_date_sequence
10. delivery_time_days

ORDER_ITEMS
Rows: 112,650
Columns (7):
1. order_id
2. order_item_id
3. product_id
4. seller_id
5. shipping_limit_date
6. price
7. freight_value

ORDER_PAYMENTS
Rows: 103,886
Columns (7):
1. order_id
2. payment_sequential
3. payment_type
4. payment_installments
5. payment_value
6. invalid_installments
7. undefined_zero_payment

ORDER_REVIEWS
Rows: 99,224
Columns (7):
1. review_id
2. order_id
3. review_score
4. review_comment_title
5. review_comment_message
6. review_creation_date
7. review_answer_timestamp

PRODUCTS
Rows: 32,951
Columns (9):
1. product_id
2. product_category_name
3. 

### 5.1 Inspect Remaining Table Columns

The initial inspection confirmed the structure of the customers and orders tables.

Since the notebook output is large, we inspect the remaining tables separately to avoid truncated output and ensure that the PostgreSQL schema matches the cleaned datasets exactly.

In [6]:
remaining_tables = [
    "order_items",
    "order_payments",
    "order_reviews",
    "products",
    "sellers",
]

for table in remaining_tables:
    df = pd.read_csv(data_path / files[table])

    print(f"\n{table.upper()}")
    print(f"Rows: {len(df):,}")
    print("Columns:")
    print("\n".join(f"  {i}. {col}" for i, col in enumerate(df.columns, 1)))


ORDER_ITEMS
Rows: 112,650
Columns:
  1. order_id
  2. order_item_id
  3. product_id
  4. seller_id
  5. shipping_limit_date
  6. price
  7. freight_value

ORDER_PAYMENTS
Rows: 103,886
Columns:
  1. order_id
  2. payment_sequential
  3. payment_type
  4. payment_installments
  5. payment_value
  6. invalid_installments
  7. undefined_zero_payment

ORDER_REVIEWS
Rows: 99,224
Columns:
  1. review_id
  2. order_id
  3. review_score
  4. review_comment_title
  5. review_comment_message
  6. review_creation_date
  7. review_answer_timestamp

PRODUCTS
Rows: 32,951
Columns:
  1. product_id
  2. product_category_name
  3. product_name_lenght
  4. product_description_lenght
  5. product_photos_qty
  6. product_weight_g
  7. product_length_cm
  8. product_height_cm
  9. product_width_cm

SELLERS
Rows: 3,095
Columns:
  1. seller_id
  2. seller_zip_code_prefix
  3. seller_city
  4. seller_state


### 5.2 Inspect Order Reviews and Product Tables

The order items and payment tables have been successfully inspected.

We now inspect the remaining review and product tables separately so that all columns are visible before designing the PostgreSQL schema.

In [7]:
for table in ["order_reviews", "products"]:
    df = pd.read_csv(data_path / files[table])

    print(f"\n{'=' * 60}")
    print(table.upper())
    print(f"{'=' * 60}")
    print(f"Rows: {len(df):,}")
    print(f"Columns ({len(df.columns)}):")

    for i, col in enumerate(df.columns, 1):
        print(f"{i}. {col}")


ORDER_REVIEWS
Rows: 99,224
Columns (7):
1. review_id
2. order_id
3. review_score
4. review_comment_title
5. review_comment_message
6. review_creation_date
7. review_answer_timestamp

PRODUCTS
Rows: 32,951
Columns (9):
1. product_id
2. product_category_name
3. product_name_lenght
4. product_description_lenght
5. product_photos_qty
6. product_weight_g
7. product_length_cm
8. product_height_cm
9. product_width_cm


## 6. Validate Data Types and Keys

Before creating the PostgreSQL schema, we validate the cleaned datasets for:

- Column data types
- Missing values
- Number of unique values
- Potential primary keys
- Foreign key columns

This helps us design a reliable relational database schema and avoid incorrect data types or broken relationships during data import.

In [8]:
for table, file in files.items():
    df = pd.read_csv(data_path / file)

    print(f"\n{'=' * 70}")
    print(table.upper())
    print(f"{'=' * 70}")

    print(df.dtypes)


CUSTOMERS
customer_id                   str
customer_unique_id            str
customer_zip_code_prefix    int64
customer_city                 str
customer_state                str
dtype: object

ORDERS
order_id                             str
customer_id                          str
order_status                         str
order_purchase_timestamp             str
order_approved_at                    str
order_delivered_carrier_date         str
order_delivered_customer_date        str
order_estimated_delivery_date        str
invalid_date_sequence               bool
delivery_time_days               float64
dtype: object

ORDER_ITEMS
order_id                   str
order_item_id            int64
product_id                 str
seller_id                  str
shipping_limit_date        str
price                  float64
freight_value          float64
dtype: object

ORDER_PAYMENTS
order_id                      str
payment_sequential          int64
payment_type                  str
payment_ins

In [9]:
key_columns = {
    "customers": ["customer_id", "customer_unique_id"],
    "orders": ["order_id", "customer_id"],
    "order_items": ["order_id", "order_item_id", "product_id", "seller_id"],
    "order_payments": ["order_id", "payment_sequential"],
    "order_reviews": ["review_id", "order_id"],
    "products": ["product_id"],
    "sellers": ["seller_id"],
}

for table, columns in key_columns.items():
    df = pd.read_csv(data_path / files[table])

    print(f"\n{'=' * 70}")
    print(table.upper())
    print(f"{'=' * 70}")

    for col in columns:
        print(
            f"{col:35} "
            f"Nulls: {df[col].isna().sum():6,} | "
            f"Unique: {df[col].nunique():7,}"
        )


CUSTOMERS
customer_id                         Nulls:      0 | Unique:  99,441
customer_unique_id                  Nulls:      0 | Unique:  96,096

ORDERS
order_id                            Nulls:      0 | Unique:  99,441
customer_id                         Nulls:      0 | Unique:  99,441

ORDER_ITEMS
order_id                            Nulls:      0 | Unique:  98,666
order_item_id                       Nulls:      0 | Unique:      21
product_id                          Nulls:      0 | Unique:  32,951
seller_id                           Nulls:      0 | Unique:   3,095

ORDER_PAYMENTS
order_id                            Nulls:      0 | Unique:  99,440
payment_sequential                  Nulls:      0 | Unique:      29

ORDER_REVIEWS
review_id                           Nulls:      0 | Unique:  98,410
order_id                            Nulls:      0 | Unique:  98,673

PRODUCTS
product_id                          Nulls:      0 | Unique:  32,951

SELLERS
seller_id                         

### 6.1 Validate Composite Keys and Relationships

Some tables contain multiple records for the same order.

For example, one order can contain multiple items and multiple payment records.

Therefore, we validate:

- Composite uniqueness of order items
- Composite uniqueness of payment records
- Foreign key relationships between tables
- Whether referenced IDs exist in their parent tables

This prevents duplicate records and broken relationships when the data is loaded into PostgreSQL.

In [10]:
# Load required tables
customers = pd.read_csv(data_path / files["customers"])
orders = pd.read_csv(data_path / files["orders"])
order_items = pd.read_csv(data_path / files["order_items"])
payments = pd.read_csv(data_path / files["order_payments"])
products = pd.read_csv(data_path / files["products"])
sellers = pd.read_csv(data_path / files["sellers"])


# 1. Order items composite key
item_duplicates = order_items.duplicated(
    subset=["order_id", "order_item_id"]
).sum()

# 2. Payment composite key
payment_duplicates = payments.duplicated(
    subset=["order_id", "payment_sequential"]
).sum()

# 3. Order → Customer relationship
missing_customers = (
    ~orders["customer_id"].isin(customers["customer_id"])
).sum()

# 4. Order Items → Orders
missing_orders = (
    ~order_items["order_id"].isin(orders["order_id"])
).sum()

# 5. Order Items → Products
missing_products = (
    ~order_items["product_id"].isin(products["product_id"])
).sum()

# 6. Order Items → Sellers
missing_sellers = (
    ~order_items["seller_id"].isin(sellers["seller_id"])
).sum()


print("Order Items duplicate composite keys:", item_duplicates)
print("Payment duplicate composite keys:", payment_duplicates)
print("Orders with missing customers:", missing_customers)
print("Order items with missing orders:", missing_orders)
print("Order items with missing products:", missing_products)
print("Order items with missing sellers:", missing_sellers)

Order Items duplicate composite keys: 0
Payment duplicate composite keys: 0
Orders with missing customers: 0
Order items with missing orders: 0
Order items with missing products: 0
Order items with missing sellers: 0


## 7. PostgreSQL Database Schema

Based on the cleaned datasets and relationship validation, we will create a relational PostgreSQL database for RetailIQ Analytics.

### Core Tables

- `customers` — Customer information
- `orders` — Order-level information
- `order_items` — Individual items within each order
- `order_payments` — Payment information
- `order_reviews` — Customer reviews
- `products` — Product information
- `sellers` — Seller information

### Relationships

```text
customers
    │
    │ customer_id
    ▼
orders
    │
    ├───────────────┬────────────────┐
    │               │                │
    ▼               ▼                ▼
order_items   order_payments   order_reviews
    │
    ├── product_id ──► products
    │
    └── seller_id ───► sellers

## 8. Create PostgreSQL Tables

We now create the PostgreSQL tables based on the validated schema.

Tables will be created in dependency order so that foreign key relationships can be established correctly.

We begin with the `customers` table because it is a parent table for `orders`.

## 9. Create the Orders Table

The `orders` table contains one record per order.

Each order belongs to a customer through `customer_id`, which references the `customers` table.

### Key Design

- `order_id` → Primary Key
- `customer_id` → Foreign Key
- Timestamp columns → `TIMESTAMP`
- `delivery_time_days` → numeric analytical field
- `invalid_date_sequence` → data-quality flag

## 10. Create the Products Table

The `products` table contains one record per product.

Each product is identified by `product_id`. Product attributes include category, physical dimensions, weight, number of photos, and text-length features.

The `product_id` will later be referenced by the `order_items` table.

## 11. Create the Sellers Table

The `sellers` table contains one record per seller.

Each seller is identified by `seller_id`. Seller location information is stored using the ZIP code prefix, city, and state.

The `seller_id` will later be referenced by the `order_items` table.

## 12. Create the Order Items Table

The `order_items` table contains one record for each product item within an order.

An order can contain multiple items, so `order_id` is not unique by itself.

The combination of `order_id` and `order_item_id` uniquely identifies each item record.

### Key Design

- `(order_id, order_item_id)` → Composite Primary Key
- `order_id` → Foreign Key to `orders`
- `product_id` → Foreign Key to `products`
- `seller_id` → Foreign Key to `sellers`
- `price` and `freight_value` → Monetary numeric fields

## 13. Create the Order Payments Table

The `order_payments` table contains payment records associated with orders.

An order can have multiple payment records, so `order_id` alone cannot uniquely identify a payment record.

The combination of `order_id` and `payment_sequential` uniquely identifies each payment record.

### Key Design

- `(order_id, payment_sequential)` → Composite Primary Key
- `order_id` → Foreign Key to `orders`
- `payment_type` → Payment method
- `payment_installments` → Number of installments
- `payment_value` → Payment amount
- `invalid_installments` → Data-quality flag
- `undefined_zero_payment` → Data-quality flag

## 14. Create the Order Reviews Table

The `order_reviews` table contains customer review information associated with orders.

Each review is identified by `review_id`.

The table stores the review score, optional review text, and review creation and response timestamps.

The `order_id` column references the corresponding order in the `orders` table.

## 15. Import Cleaned Data into PostgreSQL

The database schema has been created and validated.

We now load the cleaned CSV files from `data/processed` into PostgreSQL.

Data will be imported in dependency order so that foreign key constraints remain valid.

### Import Order

1. customers
2. products
3. sellers
4. orders
5. order_items
6. order_payments
7. order_reviews

### 15.2 Import Products

The cleaned product dataset is loaded into the PostgreSQL `products` table.

The number of imported records is then validated against the cleaned CSV dataset.

### Data Type Adjustment During Import

During the initial product data import, PostgreSQL rejected values such as `40.0` because some product attribute columns were defined as `INTEGER`.

The cleaned CSV stores these values in decimal format due to the data-cleaning/export process. Therefore, the corresponding PostgreSQL columns were changed to `NUMERIC(10,2)` to preserve the values exactly as represented in the cleaned dataset.

The source CSV was not modified.

### 15.3 Import Sellers

The cleaned seller dataset is loaded into the PostgreSQL `sellers` table.

The imported row count is then compared with the cleaned CSV dataset to validate the load.

### 15.4 Import Orders

The cleaned orders dataset is loaded into the PostgreSQL `orders` table.

The orders table contains timestamp fields and analytical data-quality fields created during the cleaning process.

After loading the data, the row count will be validated against the cleaned dataset.

### 15.5 Import Order Items

The cleaned order items dataset contains individual products purchased within each order.

Each record is uniquely identified by the composite key:

`(order_id, order_item_id)`

The table also contains foreign keys referencing:

- `orders`
- `products`
- `sellers`

The imported row count will be validated after loading.

### 15.6 Import Order Payments

The cleaned payment dataset is loaded into the PostgreSQL `order_payments` table.

An order can have multiple payment records, identified by the composite key:

`(order_id, payment_sequential)`

The imported row count will be validated against the cleaned dataset.

### 15.7 Import Order Reviews

The cleaned order reviews dataset is loaded into the PostgreSQL `order_reviews` table.

Each review is identified by `review_id` and is associated with an order through `order_id`.

After loading, the row count will be validated against the cleaned dataset.

### 15.7.1 Handle Character Encoding During Review Import

The initial review-data import failed because some review text contains characters that were not correctly interpreted under the Windows console encoding.

Since the PostgreSQL database uses UTF-8, the import is retried with the CSV encoding explicitly specified as UTF-8.

The source CSV file is not modified.

### 15.7.2 Validate Review Table Keys

The initial review import showed that `review_id` is not unique in the cleaned dataset.

Before changing the PostgreSQL primary key, we validate:

- Total review records
- Unique `review_id` values
- Unique `(review_id, order_id)` combinations
- Duplicate `(review_id, order_id)` combinations

This ensures that the PostgreSQL primary key matches the actual grain of the dataset.

In [12]:
reviews = pd.read_csv(
    data_path / files["order_reviews"]
)

print("Total rows:", len(reviews))
print("Unique review_id:", reviews["review_id"].nunique())
print(
    "Unique (review_id, order_id) pairs:",
    reviews[["review_id", "order_id"]].drop_duplicates().shape[0]
)

duplicate_pairs = (
    reviews
    .groupby(["review_id", "order_id"])
    .size()
    .reset_index(name="occurrences")
)

duplicate_pairs = duplicate_pairs[
    duplicate_pairs["occurrences"] > 1
]

print(
    "Duplicate (review_id, order_id) pairs:",
    len(duplicate_pairs)
)

duplicate_pairs.head(10)

Total rows: 99224
Unique review_id: 98410
Unique (review_id, order_id) pairs: 99224
Duplicate (review_id, order_id) pairs: 0


,review_id,order_id,occurrences


### 15.7.3 Finalize the Order Reviews Primary Key

Validation showed that `review_id` is not unique in the cleaned dataset:

- Total records: 99,224
- Unique `review_id` values: 98,410
- Unique `(review_id, order_id)` combinations: 99,224
- Duplicate `(review_id, order_id)` combinations: 0

Therefore, the actual grain of the review dataset is represented by the combination of `review_id` and `order_id`.

The PostgreSQL table will use `(review_id, order_id)` as a composite primary key.

## 16. Final Database Validation

The cleaned datasets have now been loaded into PostgreSQL.

Before beginning SQL business analysis, we perform a final validation of:

- Row counts
- Primary key uniqueness
- Foreign key integrity
- Orphan records
- Overall database completeness

This ensures that the analytical queries are executed on a structurally valid dataset.

In [13]:
expected_counts = {
    "customers": 99441,
    "products": 32951,
    "sellers": 3095,
    "orders": 99441,
    "order_items": 112650,
    "order_payments": 103886,
    "order_reviews": 99224,
}

expected_counts

{'customers': 99441,
 'products': 32951,
 'sellers': 3095,
 'orders': 99441,
 'order_items': 112650,
 'order_payments': 103886,
 'order_reviews': 99224}